# WideBind — Colab Training (D=4096, G=32, T4)
~151M params, 32 layers, PartitionedEmbed/Head (K=32, S=6),
GroupedCognitiveMirror (32 experts, k staircase 4/8/16, scalar alpha),
GroupedMLP (32 groups, expand=4), VSA hierarchical scan (2-level, fp32 guard),
Bidirectional MirrorLR, PAD/EOS masked surprisal-weighted CE.

**Перед запуском:** загрузите на Google Drive:
- `core/`, `compression/`, `scripts/` — исходники
- `token_stream_*_clean.bin` — данные
- `checkpoints/` — для resume

In [ ]:
# ─── 1. Mount Drive ───
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/widebind_data'
!ls -lh "$DRIVE"/*.bin 2>/dev/null || echo 'No .bin files'

In [ ]:
# ─── 2. Sources from Drive ───
import sys, os, shutil, glob, gc, time, math, re
import torch
import torch.nn.functional as F
import numpy as np

DRIVE = '/content/drive/MyDrive/widebind_data'
SRC = None
for d in [os.path.join(DRIVE, 'src'), DRIVE]:
    if os.path.isdir(os.path.join(d, 'core')) and os.path.isdir(os.path.join(d, 'scripts')):
        SRC = d; break
if SRC is None:
    for root, dirs, files in os.walk(DRIVE, topdown=True):
        for d in dirs: print(f'  {os.path.relpath(os.path.join(root, d), DRIVE)}/')
        break
    raise FileNotFoundError(f'core/scripts не найдены в {DRIVE}')

sys.path.insert(0, SRC); sys.path.insert(0, os.path.join(SRC, 'scripts'))
print(f'Source: {SRC}')
for sd in ['core', 'compression', 'scripts']:
    print(f'  {sd}/: {"OK" if os.path.isdir(os.path.join(SRC, sd)) else "MISSING"}')
%cd /content

In [ ]:
# ─── 3. Imports & patch ───
from core import (
    WideBindConfig, WideBindStack, MirrorLRScheduler,
    AdaptiveController, sparse_block_codes, dct_basis
)
from compression import FCF_CPR
from compression.fcf_cpr import FCF_CPR as FCP, dequantize_tensor

def _decompress(compressed, meta, cfg):
    sd = {}
    V = dct_basis(cfg.D); codes = sparse_block_codes(cfg.vocab, cfg.code_dim, cfg.code_sparsity)
    for k, v in compressed.items():
        info = meta.get(k)
        if info is None: sd[k] = v.clone()
        elif info[0] == 'uniform8':
            sd[k] = dequantize_tensor(v, info[3], info[4], info[2])
        else: sd[k] = v.detach().expand(info[1]).clone().to(info[2])
    for i in range(cfg.n_layers): sd[f'layers.{i}.V_dct'] = V.clone()
    sd['embed.codes'] = sd['lm_head.codes'] = codes.clone()
    return sd
FCP.decompress_sd = lambda s, c, m, cfg: _decompress(c, m, cfg)
print('FCF_CPR patched')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# ─── 4. Config ───
cfg = WideBindConfig(
    D=4096, n_layers=32, bind_K=32,
    mlp_groups=32, mlp_expand=4,
    code_dim=32, code_sparsity=6, mirror_k=32,
    seq_len=64, lr=3e-4, max_steps=300000,
    warmup_steps=2000, scheduler='mirror',
    grad_clip=0.5, conv_kernel=48,
    data_dir=DRIVE,
    save_dir=os.path.join(DRIVE, 'checkpoints'),
    log_dir=os.path.join(DRIVE, 'logs'),
    lambda_lr_hierarchy=True,
    gate_l1_weight=0.001, reinforce_weight=0.01,
    balance_weight=0.01, diversity_weight=0.001,
    nuclear_weight=1e-5, orth_weight=1e-4,
)
print(f'D={cfg.D} L={cfg.n_layers} G={cfg.mlp_groups} K={cfg.bind_K}')

In [ ]:
# ─── 5. Model (CPU) ───
model = WideBindStack(cfg)
n = model.param_count()
print(f'Params: {n:,} ({n/1e6:.2f}M) — на CPU')

x = torch.randint(0, 50000, (1, 16))
h = model.embed_tokens(x); out, state, gs = model(h)
assert out.shape == (1, 16, 4096), f'Bad shape: {out.shape}'
print('Forward: OK (CPU)')

In [ ]:
# ─── 6. Data streams ───
stream_files = sorted(glob.glob(os.path.join(DRIVE, 'token_stream_*_clean.bin')))
if not stream_files:
    stream_files = sorted(glob.glob(os.path.join(DRIVE, 'token_stream_*.bin')))
streams = [np.memmap(f, dtype=np.uint16, mode='r') for f in stream_files]
total = sum(len(s) for s in streams)
print(f'Streams: {len(streams)} files, {total:,} tokens')

class StreamReader:
    def __init__(self, streams, seq_len, bs):
        self.streams = streams; self.S = seq_len; self.B = bs
        self.si = 0; self.off = 0
    def get_batch(self):
        s = self.streams[self.si]
        need = self.B * self.S + 1
        if self.off + need > len(s):
            self.off = 0; self.si = (self.si + 1) % len(self.streams)
            s = self.streams[self.si]
        ch = s[self.off:self.off+need]
        x = torch.from_numpy(ch[:self.B*self.S].copy()).long().view(self.B, self.S)
        y = torch.from_numpy(ch[1:self.B*self.S+1].copy()).long().view(self.B, self.S)
        self.off += self.B * self.S
        return x, y
    def reset(self): self.off = 0

In [ ]:
# ─── 7. Batch size + GPU зачистка ───
B = 2
cfg.batch_size = B
reader = StreamReader(streams, cfg.seq_len, B)
torch.cuda.empty_cache(); gc.collect()
print(f'Batch size: {B}')
print(f'GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} / {torch.cuda.mem_get_info()[1]/1e9:.2f} GB')

In [ ]:
# ─── 8. Model to GPU + Optimizer ───
torch.cuda.empty_cache(); gc.collect()
print(f'GPU before: {torch.cuda.mem_get_info()[0]/1e9:.2f} free')
model = model.to(device)
groups = model.param_groups()
optimizer = torch.optim.AdamW(groups, betas=(0.9, 0.95))
scheduler = MirrorLRScheduler(model, optimizer, cfg=cfg)
scaler = torch.cuda.amp.GradScaler()
accum = getattr(cfg, 'accum_steps', 1)
print(f'GPU after: {torch.cuda.mem_get_info()[0]/1e9:.2f} free')
print(f'Optimizer ready, accum={accum}')

In [ ]:
# ─── 9. Resume ───
start_step = 0; best_val = float('inf')
os.makedirs(cfg.save_dir, exist_ok=True)

ckpts = sorted(glob.glob(os.path.join(cfg.save_dir, '*.pt')),
               key=lambda p: int(re.search(r'step_(\d+)', os.path.basename(p)).group(1))
               if re.search(r'step_(\d+)', os.path.basename(p)) else 0)
if ckpts:
    ckpt = torch.load(ckpts[-1], map_location='cpu', weights_only=False)
    if 'model' in ckpt:
        sd = FCP.decompress_sd(None, ckpt.get('compressed', {}),
                               ckpt.get('meta', {}), cfg) if 'compressed' in ckpt else ckpt['model']
        model.load_state_dict(sd, strict=False)
    if 'optimizer' in ckpt:
        try: optimizer.load_state_dict(ckpt['optimizer'])
        except: pass
    if 'scheduler' in ckpt:
        try: scheduler.load_state_dict(ckpt['scheduler'])
        except: pass
    start_step = ckpt.get('step', 0)
    best_val = ckpt.get('best_val_loss', float('inf'))
    print(f'Resumed: step {start_step}, best val_loss {best_val:.4f}')
    del ckpt, sd; gc.collect()
else:
    print('Fresh start')

In [ ]:
# ─── 10. Eval ───
@torch.no_grad()
def evaluate():
    model.eval()
    total, cnt = 0.0, 0
    n = int(min(100, sum(len(s) for s in streams) // (B * cfg.seq_len) // len(streams)))
    for si, s in enumerate(streams):
        off = max(len(s) // 4, B * cfg.seq_len + 1); state = None; gs = None
        for _ in range(n):
            if off + B * cfg.seq_len + 1 > len(s): off = 0; break
            ch = s[off:off+B*cfg.seq_len+1]
            x = torch.from_numpy(ch[:B*cfg.seq_len].copy()).long().view(B, cfg.seq_len).to(device)
            y = torch.from_numpy(ch[1:B*cfg.seq_len+1].copy()).long().view(B, cfg.seq_len).to(device)
            off += B * cfg.seq_len
            h = model.embed_tokens(x); out, state, gs = model(h, state, global_state=gs, adaptive=False)
            total += model.compute_loss(out, y).item(); cnt += 1
            if _ % 25 == 24: state = None; gs = None
        del state, gs; gc.collect()
    model.train()
    return total / max(cnt, 1)

In [ ]:
# ─── 11. Training Loop ───
state = None; gs = None; t0 = time.time(); tokens = 0

def _detach(x):
    if x is None: return None
    if isinstance(x, torch.Tensor): return x.detach()
    if isinstance(x, dict): return {k: _detach(v) for k, v in x.items()}
    if isinstance(x, (tuple, list)): return type(x)(_detach(v) for v in x)
    return x

print(f'Training: {start_step} → {cfg.max_steps}')
print(f'GPU: {torch.cuda.mem_get_info()[0]/1e9:.2f} free / {torch.cuda.mem_get_info()[1]/1e9:.2f} total')
try:
    for step in range(start_step, cfg.max_steps):
        x, y = reader.get_batch()
        x, y = x.to(device), y.to(device)

        if (y[:, -1] == 2).any():
            if state is not None: state = _detach(state)

        with torch.amp.autocast('cuda'):
            h = model.embed_tokens(x)
            out, state, gs = model(h, state, global_state=gs)
            loss = model.compute_loss(out, y) / accum

        # Detach state — не копим графы между шагами
        state = _detach(state)
        gs = _detach(gs)

        scaler.scale(loss).backward()
        tokens += B * cfg.seq_len

        if (step + 1) % accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        if step % cfg.log_interval == 0:
            dt = time.time() - t0
            try:
                idiff = torch.stack([(1.0 - l.mirror.alpha.data).abs().mean() for l in model.layers]).mean().item()
                gvar = torch.stack([l.mirror._last_gates.var() for l in model.layers]).mean().item()
                ls_var = torch.stack([l.mirror.log_scale.data.var() for l in model.layers]).mean().item()
            except: idiff = gvar = ls_var = 0.0
            lr = scheduler.get_last_lr()[0]
            mem = torch.cuda.max_memory_allocated() / 1e9 if device == 'cuda' else 0
            print(f'step={step:>6} loss={loss.item()*accum:.4f} |1-a|={idiff:.4f} '
                  f'g_var={gvar:.4f} ls_var={ls_var:.4f} lr={lr:.2e} tok/s={tokens/dt:.0f} mem={mem:.1f}GB')
            if device == 'cuda': torch.cuda.reset_peak_memory_stats()

        if step > 0 and step % cfg.eval_interval == 0:
            vl = evaluate()
            print(f'EVAL step={step}: val_loss={vl:.4f} ppl={math.exp(vl):.2f}')
            scheduler.report_val_loss(vl)
            torch.cuda.empty_cache(); gc.collect()
            if vl < best_val:
                best_val = vl
                torch.save({'step': step, 'model': model.state_dict(),
                           'best_val_loss': best_val, 'cfg': cfg},
                           os.path.join(cfg.save_dir, 'best.pt'))
                print(f'  New best!')

        if step > 0 and step % cfg.save_interval == 0:
            torch.save({'step': step, 'model': model.state_dict(),
                       'optimizer': optimizer.state_dict(),
                       'scheduler': scheduler.state_dict(),
                       'best_val_loss': best_val, 'cfg': cfg},
                       os.path.join(cfg.save_dir, f'step_{step}.pt'))
            print(f'Checkpoint saved')

except KeyboardInterrupt:
    torch.save({'step': step, 'model': model.state_dict(),
               'optimizer': optimizer.state_dict(),
               'scheduler': scheduler.state_dict(),
               'best_val_loss': best_val, 'cfg': cfg},
               os.path.join(cfg.save_dir, f'interrupt_step_{step}.pt'))
    print(f'\nInterrupted. Saved step_{step}.pt')

In [ ]:
# ─── 12. Analyze latest checkpoint ───
from scripts.analyze_checkpoint import generate_report

latest = sorted(glob.glob(os.path.join(cfg.save_dir, 'step_*.pt')),
                key=lambda p: int(re.search(r'step_(\d+)', os.path.basename(p)).group(1)))
if latest:
    path = generate_report(latest[-1])
    dst = os.path.join(cfg.log_dir, os.path.basename(path))
    os.makedirs(cfg.log_dir, exist_ok=True)
    shutil.copy2(path, dst)
    print(f'Report: {dst}')
    from IPython.display import HTML, display
    with open(path) as f: display(HTML(f.read()))